In [4]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

In [2]:
df = pd.read_parquet("基础数据.parquet")

# 日期转为datetime，股票代码保持字符串
df["日期"] = pd.to_datetime(df["日期"])
df = df.sort_values(["股票代码", "日期"]).reset_index(drop=True)
df.head()

,股票代码,日期,原始股价,成交金额,总市值,换手率,二十日波动率,复权后收盘价,复盘后开盘价
0,000001,2012-01-04,0.859994,1.665508,1.674848,-0.891903,NaN,4.663827,4.796112
1,000001,2012-01-05,0.981040,1.699636,1.674641,-0.367109,NaN,4.734584,4.663827
2,000001,2012-01-06,0.951873,1.658025,1.678179,-1.107675,-1.648875,4.722278,4.722278
3,000001,2012-01-09,0.934647,1.667123,1.674899,-0.929981,-1.661173,4.854564,4.725355
4,000001,2012-01-20,1.044570,1.721605,1.675180,0.803932,NaN,5.162204,5.146822


In [ ]:
# 基础信息
print("形状:", df.shape)
print("股票数量:", df["股票代码"].nunique())
print("日期范围:", df["日期"].min().date(), "~", df["日期"].max().date())
print("交易日数量:", df["日期"].nunique())
df.info()

形状: (5112322, 9)
股票数量: 3077
日期范围: 2012-01-04 ~ 2020-12-31
交易日数量: 2188
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5112322 entries, 0 to 5112321
Data columns (total 9 columns):
 #   Column  Dtype         
---  ------  -----         
 0   股票代码    object        
 1   日期      datetime64[ns]
 2   原始股价    float64       
 3   成交金额    float64       
 4   总市值     float64       
 5   换手率     float64       
 6   二十日波动率  float64       
 7   复权后收盘价  float64       
 8   复盘后开盘价  float64       
dtypes: datetime64[ns](1), float64(7), object(1)
memory usage: 351.0+ MB


In [3]:
# 有缺失值的日期表：每个交易日缺失二十日波动率的股票数量
miss_date = df.groupby("日期")["二十日波动率"].apply(lambda s: s.isna().sum())
daily_stocks = df.groupby("日期").size()  # 每日股票数，用于计算当日缺失率

miss_table = pd.DataFrame({
    "缺失股票数": miss_date,
    "当日股票数": daily_stocks,
})
miss_table["当日缺失率(%)"] = (miss_table["缺失股票数"] / miss_table["当日股票数"] * 100).round(2)

# 仅保留存在缺失的日期
miss_table = miss_table[miss_table["缺失股票数"] > 0].sort_values("日期")
print("存在缺失的交易日数:", len(miss_table), "/ 总交易日:", len(miss_date))
miss_table

存在缺失的交易日数: 2188 / 总交易日: 2188


,缺失股票数,当日股票数,当日缺失率(%)
日期,,,
2012-01-04,496,1950,25.44
2012-01-05,498,1946,25.59
2012-01-06,512,1956,26.18
2012-01-09,512,1953,26.22
2012-01-10,512,1958,26.15
...,...,...,...
2020-12-25,42,3008,1.40
2020-12-28,46,3011,1.53
2020-12-29,46,3013,1.53


In [5]:
def backtest(train, test, X_cols):
    """训练线性回归并回测，返回(模型, 每日IC序列, 测试集预测值)。"""
    reg = LinearRegression()
    reg.fit(train[X_cols], train["Y"])
    pred = reg.predict(test[X_cols])

    ic_df = pd.DataFrame({"日期": test["日期"], "Y": test["Y"], "pred": pred})
    ic_daily = ic_df.groupby("日期").apply(
        lambda g: g["pred"].corr(g["Y"]) if len(g) >= 2 else np.nan,
        include_groups=False,
    )
    return reg, ic_daily, pred

In [7]:
# 基准
df["未来5日收盘"] = df.groupby("股票代码")["复权后收盘价"].shift(-5)
df["次日开盘"] = df.groupby("股票代码")["复盘后开盘价"].shift(-1)
df["Y"] = (df["未来5日收盘"] / df["次日开盘"]) / 5

X_cols = ["换手率", "总市值", "成交金额", "原始股价", "二十日波动率"]
df[X_cols + ["Y"]].head()

model_df = df[X_cols + ["Y", "日期", "股票代码"]].dropna().reset_index(drop=True)

train = model_df[model_df["日期"].dt.year <= 2018]   # 2012-2018
test = model_df[model_df["日期"].dt.year >= 2019]    # 2019-2020
print("训练集:", train.shape, " 测试集:", test.shape)

reg, ic_daily, test_pred = backtest(train, test, X_cols)

print("截距:", reg.intercept_)
print("回归系数:")
print(pd.Series(reg.coef_, index=X_cols))

print("\n每日 IC 均值:", round(ic_daily.mean(), 6))
print("每日 IC 标准差:", round(ic_daily.std(), 6))
print("IC > 0 的天数占比:", round((ic_daily > 0).mean(), 4))

训练集: (3412696, 8)  测试集: (1369523, 8)
截距: 0.2008031464970111
回归系数:
换手率      -0.000028
总市值      -0.000175
成交金额     -0.000428
原始股价     -0.000088
二十日波动率    0.000094
dtype: float64

每日 IC 均值: 0.011705
每日 IC 标准差: 0.115055
IC > 0 的天数占比: 0.5539
